# Get list of entrants

In [1]:
import json
from gql import gql, Client
from gql.transport.requests import RequestsHTTPTransport

In [2]:
import os

auth_token = os.environ['SMASHGG_TOKEN']
api_version = 'alpha'

In [3]:
transport = RequestsHTTPTransport(
    url=f'https://api.start.gg/gql/{api_version}',
    headers={'Authorization': f'Bearer {auth_token}'},
    use_json=True,
)

client = Client(transport=transport, fetch_schema_from_transport=False)


In [4]:
# Define the API endpoint and query for tournament details
tournament_slug = "ngpr"
query = gql("""
query TournamentQuery($slug: String!) {
    tournament(slug: $slug) {
        id
        name
        city
        state
        countryCode
        startAt
        endAt
        events {
            id
            name
            numEntrants
        }
    }
}
""")

variables = {"slug": tournament_slug}

# Execute the query
try:
        tournament_result = client.execute(query, variable_values=variables)
        print(json.dumps(tournament_result, indent=2))
except Exception as e:
        print("Error fetching tournament details:", e)

{
  "tournament": {
    "id": 808121,
    "name": "New Game Plus Revival 8.13",
    "city": "Boston",
    "state": 3,
    "countryCode": "US",
    "startAt": 1752616800,
    "endAt": 1752638340,
    "events": [
      {
        "id": 1417057,
        "name": "Project+ Singles (Free!)",
        "numEntrants": 1
      },
      {
        "id": 1417055,
        "name": "Melee Singles",
        "numEntrants": 30
      },
      {
        "id": 1417056,
        "name": "Melee Redemption",
        "numEntrants": 9
      }
    ]
  }
}


In [5]:
# Extract the event id for "Melee Singles"
events = tournament_result['tournament']['events']
melee_singles_id = next((event['id'] for event in events if event['name'] == "Melee Singles"), None)
print("Melee Singles Event ID:", melee_singles_id)

Melee Singles Event ID: 1417055


In [6]:
# Query to get entrants for Melee Singles
entrants_query = gql("""
query EventEntrants($eventId: ID!) {
    event(id: $eventId) {
        entrants(query: {page: 1, perPage: 512}) {
            nodes {
                id
                name
                participants {
                    gamerTag
                }
            }
        }
    }
}
""")

entrants_variables = {"eventId": melee_singles_id}

try:
        entrants_result = client.execute(entrants_query, variable_values=entrants_variables)
        print(json.dumps(entrants_result, indent=2))
except Exception as e:
        print("Error fetching entrants:", e)

{
  "event": {
    "entrants": {
      "nodes": [
        {
          "id": 20701899,
          "name": "hc | pluto",
          "participants": [
            {
              "gamerTag": "pluto"
            }
          ]
        },
        {
          "id": 20701804,
          "name": "MAIF",
          "participants": [
            {
              "gamerTag": "MAIF"
            }
          ]
        },
        {
          "id": 20701789,
          "name": "hc | kraft",
          "participants": [
            {
              "gamerTag": "kraft"
            }
          ]
        },
        {
          "id": 20701782,
          "name": "BonkCushy",
          "participants": [
            {
              "gamerTag": "BonkCushy"
            }
          ]
        },
        {
          "id": 20700995,
          "name": "ThunderPaste",
          "participants": [
            {
              "gamerTag": "ThunderPaste"
            }
          ]
        },
        {
          "id": 20700993,
    

In [13]:
import polars as pl

# Extract entrant data and flatten participants' gamerTags

entrant_nodes = entrants_result['event']['entrants']['nodes']
data = []
for entrant in entrant_nodes:
    entrant_id = entrant['id']
    entrant_name = entrant['name']
    # There may be multiple participants per entrant; join their gamerTags with comma
    gamer_tags = [p['gamerTag'] for p in entrant.get('participants', [])]
    gamer_tag = ', '.join(gamer_tags)
    data.append({'id': entrant_id, 'name': entrant_name, 'gamertag': gamer_tag})

entrants = pl.DataFrame(data)
entrants

id,name,gamertag
i64,str,str
20701899,"""hc | pluto""","""pluto"""
20701804,"""MAIF""","""MAIF"""
20701789,"""hc | kraft""","""kraft"""
20701782,"""BonkCushy""","""BonkCushy"""
20700995,"""ThunderPaste""","""ThunderPaste"""
…,…,…
20687727,"""AUNTIE WOW | bleop""","""bleop"""
20687556,"""Ant""","""Ant"""
20686872,"""bfu""","""bfu"""


In [19]:
entrant_gamertags = entrants.select(pl.col('gamertag').str.to_lowercase()).to_series().to_list()

# collect seeding

In [84]:
player_ratings = pl.read_parquet('data/player-ratings.parquet')

In [87]:
pl.Config(tbl_rows=100)
entrant_ratings = (
    entrants
    .with_columns(pl.col('gamertag').str.to_lowercase())
    .join(
        player_ratings
            .with_columns(pl.col('tag').str.split(' | ').list.last().str.to_lowercase()),
        how='full',
        left_on='gamertag',
        right_on='tag',
    )
    #.filter(
    #    (pl.col('tag').str.to_lowercase() == pl.col('gamertag').str.to_lowercase()) |
    #    pl.col('gamertag').is_null()
    #)
    .filter(pl.col('gamertag').is_not_null())
    .sort('rating', descending=True)
    .group_by('id').first()
    .sort('rating', descending=True)
)
display(entrant_ratings)

id,name,gamertag,url,rating,tag
i64,str,str,str,f64,str
20700362,"""regEx""","""regex""","""/league/nemelee/player/1D8B9E3…",36.887306,"""regex"""
20687556,"""Ant""","""ant""","""/league/nemelee/player/5DBCC34…",35.158962,"""ant"""
20699715,"""Karils | Conflict""","""conflict""","""/league/nemelee/player/38A911E…",34.01907,"""conflict"""
20698528,"""hc | saucymain""","""saucymain""","""/league/nemelee/player/2407F41…",33.159399,"""saucymain"""
20696744,"""Sweat""","""sweat""","""/league/nemelee/player/DF3CE0B…",33.128794,"""sweat"""
20701789,"""hc | kraft""","""kraft""","""/league/nemelee/player/0CDB11B…",31.636772,"""kraft"""
20701782,"""BonkCushy""","""bonkcushy""","""/league/nemelee/player/1056255…",30.556306,"""bonkcushy"""
20700995,"""ThunderPaste""","""thunderpaste""","""/league/nemelee/player/E545213…",30.168946,"""thunderpaste"""
20699209,"""Swartzy""","""swartzy""","""/league/nemelee/player/2D23DC1…",29.017643,"""swartzy"""


In [ ]:
entrant_seeding = (
    entrant_ratings
    .filter(pl.col('rating').is_not_null())
    .sort('rating', descending=True)
    .with_row_index('seed_num')
    .with_columns(pl.col('seed_num') + 1)
)
entrant_seeding

seed_num,id,name,gamertag,url,rating,tag
u32,i64,str,str,str,f64,str
1,20700362,"""regEx""","""regex""","""/league/nemelee/player/1D8B9E3…",36.301434,"""regex"""
2,20687556,"""Ant""","""ant""","""/league/nemelee/player/5DBCC34…",34.819169,"""ant"""
3,20699715,"""Karils | Conflict""","""conflict""","""/league/nemelee/player/38A911E…",32.603542,"""conflict"""
4,20698528,"""hc | saucymain""","""saucymain""","""/league/nemelee/player/2407F41…",31.66636,"""saucymain"""
5,20696744,"""Sweat""","""sweat""","""/league/nemelee/player/DF3CE0B…",31.613687,"""sweat"""
6,20701789,"""hc | kraft""","""kraft""","""/league/nemelee/player/0CDB11B…",31.226735,"""kraft"""
7,20701782,"""BonkCushy""","""bonkcushy""","""/league/nemelee/player/1056255…",30.252516,"""bonkcushy"""
8,20700995,"""ThunderPaste""","""thunderpaste""","""/league/nemelee/player/E545213…",29.386891,"""thunderpaste"""
9,20668097,"""zaubermaus""","""zaubermaus""","""/league/nemelee/player/66DEE19…",28.319641,"""zaubermaus"""


# Export seeding to start gg

In [53]:
# Query to get phase IDs for Melee Singles event
phases_query = gql("""
query EventPhases($eventId: ID!) {
    event(id: $eventId) {
        phases {
            id
            name
            numSeeds
        }
    }
}
""")

phases_variables = {"eventId": melee_singles_id}

try:
        phases_result = client.execute(phases_query, variable_values=phases_variables)
        print(json.dumps(phases_result, indent=2))
except Exception as e:
        print("Error fetching phases:", e)

phase_id = phases_result['event']['phases'][0]['id']
print(f'phase id: {phase_id}')

{
  "event": {
    "phases": [
      {
        "id": 2021471,
        "name": "Bracket",
        "numSeeds": 30
      }
    ]
  }
}
phase id: 2021471


In [58]:
# Prepare seed mapping from entrant_ratings for the phase
seed_mapping = []
for row in entrant_seeding.iter_rows(named=True):
    seed_mapping.append({
        "seedId": row["id"],
        "seedNum": row['seed_num'],
    })

print(f"Importing {len(seed_mapping)} seeds to phase {phase_id}...")

mutation = gql("""
mutation UpdatePhaseSeeding($phaseId: ID!, $seedMapping: [UpdatePhaseSeedInfo]!) {
  updatePhaseSeeding(phaseId: $phaseId, seedMapping: $seedMapping) {
    id
  }
}
""")

params = {
    "phaseId": phase_id,
    "seedMapping": seed_mapping,
}

try:
    result = client.execute(mutation, variable_values=params)
    print('Success!')
    print(result)
except Exception as e:
    print('Error:', e)

Importing 23 seeds to phase 2021471...
Error: {'message': 'Cannot modify seeds in started pools', 'extensions': {'category': 'validation'}, 'locations': [{'line': 2, 'column': 3}], 'path': ['updatePhaseSeeding'], 'type': 'validation', 'fields': None, 'fieldErrors': []}
